In [ ]:
import os
import pandas as pd
import numpy as np
from datetime import datetime as dt
from sklearn.model_selection import KFold
from sklearn.preprocessing import MinMaxScaler
from wifiplotting import *
from modeling import *
import dill

sequoia = pd.read_csv('../data/sequoia_sets.csv')

TL_CORNER = [37.430582, -122.173904]
BR_CORNER = [37.42705, -122.169413]

In [ ]:
osm_context = OSMPlotContext.from_bounds(
    init_lons=[BR_CORNER[1], TL_CORNER[1]], init_lats=[BR_CORNER[0], TL_CORNER[0]],
    pad_fraction=0.0
)

with open('data/osm_context.pkl', 'wb') as f:
    dill.dump(osm_context, f)

In [ ]:
sequoia_nonan = sequoia[sequoia['rssi_set'].notna()].copy().reset_index()
time0 = dt.strptime(sequoia['timestamp'][0], "%Y-%m-%d %H:%M:%S.%f%z")
sequoia_nonan['t'] = sequoia_nonan['timestamp'].apply(
    lambda x: (dt.strptime(x, "%Y-%m-%d %H:%M:%S.%f%z") - time0).total_seconds() / (60 * 60))

scaler = MinMaxScaler()
scaler.fit(pd.DataFrame(np.stack([TL_CORNER[::-1], BR_CORNER[::-1]]), columns=['longitude', 'latitude']))
# scaler.fit(sequoia_nonan[['longitude', 'latitude']])
X_train = pd.DataFrame()
X_train[['longitude', 'latitude']] = scaler.transform(sequoia_nonan[['longitude', 'latitude']])
X_train['indoor'] = sequoia_nonan['indoor'].astype(float).values
X_train['t'] = sequoia_nonan['t'].values

ap_categories = np.sort(sequoia_nonan['ap'].astype(str).unique())
X_train['ap_code'] = pd.Categorical(
    sequoia_nonan['ap'].astype(str),
    categories=ap_categories,
).codes.astype(float)

y_train = sequoia_nonan['rssi_set']

X_train = np.array(X_train)
y_train = np.array(y_train)
obs_count = sequoia_nonan['rssi_n_valid'].to_numpy()
obs_sse = sequoia_nonan['rssi_within_sse'].fillna(0.0).to_numpy()

np.save('data/X_train.npy', X_train)
np.save('data/y_train.npy', y_train)
np.save('data/obs_count.npy', obs_count)
np.save('data/obs_sse.npy', obs_sse)
np.save('data/ap_categories.npy', ap_categories)

In [ ]:
np.save('data/coord_train.npy', sequoia_nonan[['longitude', 'latitude']].to_numpy())

wlon_train, wlat_train = osm_context.to_world(sequoia_nonan.longitude, sequoia_nonan.latitude)

np.save('data/world_train.npy', np.stack([wlon_train, wlat_train]).T)

In [ ]:
grid_width = 100

x_new = np.linspace(0, 1, grid_width)
y_new = np.linspace(0, 1, grid_width)

grid_points = np.stack(np.meshgrid(x_new, y_new), axis=-1).reshape(-1, 2)

coord_grid = scaler.inverse_transform(np.concatenate([grid_points], axis=-1))
long_new, lat_new = coord_grid.T
wlon_test, wlat_test = osm_context.to_world(long_new, lat_new)

z_new = osm_context.contains_building(long_new, lat_new).reshape(-1,1)
t_new = np.ones((grid_points.shape[0], 1)) * 1000

ap_new = np.empty(grid_points.shape[0], dtype=X_train.dtype)
chunk_size = 4096
for start in range(0, grid_points.shape[0], chunk_size):
    stop = min(start + chunk_size, grid_points.shape[0])
    diff = grid_points[start:stop, None, :] - X_train[None, :, :2]
    nearest_idx = np.argmin(np.sum(diff * diff, axis=2), axis=1)
    ap_new[start:stop] = X_train[nearest_idx, 4]
ap_new = ap_new.reshape(-1, 1)

X_new = np.concatenate([grid_points, z_new, t_new, ap_new], axis=-1)

np.save('data/X_test.npy', X_new)
np.save('data/coord_test.npy', coord_grid)
np.save('data/world_test.npy', np.stack([wlon_test, wlat_test]).T)

In [ ]:
geo_train, geo_val = geographic_train_test_split(sequoia_nonan, random_state=0)
geo_tr_idx = geo_train.index
geo_val_idx = geo_val.index

np.save('data/geo_X_train.npy', X_train[geo_tr_idx])
np.save('data/geo_y_train.npy', y_train[geo_tr_idx])
np.save('data/geo_obs_count_train.npy', obs_count[geo_tr_idx])
np.save('data/geo_obs_sse_train.npy', obs_sse[geo_tr_idx])
np.save('data/geo_X_val.npy', X_train[geo_val_idx])
np.save('data/geo_y_val.npy', y_train[geo_val_idx])
np.save('data/geo_obs_count_val.npy', obs_count[geo_val_idx])
np.save('data/geo_obs_sse_val.npy', obs_sse[geo_val_idx])

In [ ]:
K = 5
CV_RANDOM_STATE = 0

coord_train = sequoia_nonan[['longitude', 'latitude']].to_numpy()
world_train = np.stack([wlon_train, wlat_train]).T

def save_cv_fold(out_dir, split_idx, train_idx, test_idx):
    os.makedirs(out_dir, exist_ok=True)
    np.save(os.path.join(out_dir, f'X_train_{split_idx}.npy'), X_train[train_idx])
    np.save(os.path.join(out_dir, f'y_train_{split_idx}.npy'), y_train[train_idx])
    np.save(os.path.join(out_dir, f'obs_count_train_{split_idx}.npy'), obs_count[train_idx])
    np.save(os.path.join(out_dir, f'obs_sse_train_{split_idx}.npy'), obs_sse[train_idx])
    np.save(os.path.join(out_dir, f'coord_train_{split_idx}.npy'), coord_train[train_idx])
    np.save(os.path.join(out_dir, f'world_train_{split_idx}.npy'), world_train[train_idx])
    np.save(os.path.join(out_dir, f'train_idx_{split_idx}.npy'), train_idx)

    np.save(os.path.join(out_dir, f'X_test_{split_idx}.npy'), X_train[test_idx])
    np.save(os.path.join(out_dir, f'y_test_{split_idx}.npy'), y_train[test_idx])
    np.save(os.path.join(out_dir, f'obs_count_test_{split_idx}.npy'), obs_count[test_idx])
    np.save(os.path.join(out_dir, f'obs_sse_test_{split_idx}.npy'), obs_sse[test_idx])
    np.save(os.path.join(out_dir, f'coord_test_{split_idx}.npy'), coord_train[test_idx])
    np.save(os.path.join(out_dir, f'world_test_{split_idx}.npy'), world_train[test_idx])
    np.save(os.path.join(out_dir, f'test_idx_{split_idx}.npy'), test_idx)


rand_cv_dir = 'cv/rand'
geo_cv_dir = 'cv/geo'

os.makedirs(rand_cv_dir, exist_ok=True)
os.makedirs(geo_cv_dir, exist_ok=True)
np.save(os.path.join(rand_cv_dir, 'ap_categories.npy'), ap_categories)
np.save(os.path.join(geo_cv_dir, 'ap_categories.npy'), ap_categories)

row_idx = np.arange(X_train.shape[0])
rand_folds = KFold(n_splits=K, shuffle=True, random_state=CV_RANDOM_STATE)
for split_idx, (train_idx, test_idx) in enumerate(rand_folds.split(row_idx)):
    save_cv_fold(rand_cv_dir, split_idx, train_idx, test_idx)

geo_folds = geographic_kfold_split(sequoia_nonan, K=K, random_state=CV_RANDOM_STATE)
for split_idx, (geo_train, geo_test) in enumerate(geo_folds):
    save_cv_fold(
        geo_cv_dir,
        split_idx,
        geo_train.index.to_numpy(dtype=int),
        geo_test.index.to_numpy(dtype=int),
    )